[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/02_Introduction_to_ONNX/01_What_is_ONNX/What_is_ONNX_Deep_Dive.ipynb)

# 1.1 What is ONNX? — Deep Dive

## Table of Contents
1. [The N×M Interoperability Problem](#section-1)
2. [Historical Context and Motivation](#section-2)
3. [Formal Definition of ONNX](#section-3)
4. [The Computation Graph — Mathematical Foundation](#section-4)
5. [ONNX as a Universal Intermediate Representation](#section-5)
6. [Visualization: The Framework-Runtime Matrix](#section-6)
7. [Worked Example: A Simple Graph](#section-7)
8. [The ONNX Operator Algebra](#section-8)
9. [Key Properties and Guarantees](#section-9)
10. [Summary and Connections](#section-10)

<a id='section-1'></a>
## Section 1: The N×M Interoperability Problem — Formal Definition

### The Problem Statement

In the machine learning ecosystem, we face a fundamental combinatorial explosion in tooling:

**Given:**
- A set of $N$ training frameworks: $\mathcal{F} = \{f_1, f_2, \ldots, f_N\}$ (PyTorch, TensorFlow, JAX, ...)
- A set of $M$ deployment targets: $\mathcal{D} = \{d_1, d_2, \ldots, d_M\}$ (TensorRT, OpenVINO, CoreML, ...)

**Without ONNX:** We need $N \times M$ custom converters:

$$C_{\text{direct}} = |\mathcal{F}| \times |\mathcal{D}| = N \cdot M$$

**With ONNX as a hub:** We need only $N + M$ converters:

$$C_{\text{ONNX}} = |\mathcal{F}| + |\mathcal{D}| = N + M$$

The **savings ratio** grows with ecosystem size:

$$\text{Savings} = \frac{N \cdot M}{N + M} = \frac{N \cdot M}{N + M}$$

For $N = 5$ frameworks and $M = 8$ runtimes: direct requires $40$ converters vs. ONNX's $13$ — a $3\times$ reduction. As the ecosystem grows, this ratio approaches $\min(N, M) / 2$.

### Architecture Comparison

```
WITHOUT ONNX (N×M direct converters):       WITH ONNX (N+M hub converters):

┌──────────┐    ┌──────────────┐            ┌──────────┐
│ PyTorch  │───▶│  TensorRT    │            │ PyTorch  │──┐
│          │───▶│  OpenVINO    │            │          │  │
│          │───▶│  CoreML      │            └──────────┘  │
└──────────┘    └──────────────┘            ┌──────────┐  │   ┌──────┐
┌──────────┐    ┌──────────────┐            │   TF     │──┤   │      │
│   TF     │───▶│  TensorRT    │            │          │  ├──▶│ ONNX │──┐
│          │───▶│  OpenVINO    │            └──────────┘  │   │      │  │
│          │───▶│  CoreML      │            ┌──────────┐  │   └──────┘  │
└──────────┘    └──────────────┘            │   JAX    │──┘             │
┌──────────┐    ┌──────────────┐            └──────────┘       ┌───────┴────────┐
│   JAX    │───▶│  TensorRT    │                               │                │
│          │───▶│  OpenVINO    │                          ┌────▼─────┐  ┌───────▼──────┐
│          │───▶│  CoreML      │                          │ TensorRT │  │   OpenVINO   │
└──────────┘    └──────────────┘                          └──────────┘  └──────────────┘

  Edges: N × M = 9                            Edges: N + M = 3 + 2 = 5
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualize the N×M problem vs ONNX hub solution
N_values = np.arange(2, 15)
M_values = np.arange(2, 15)
N, M = np.meshgrid(N_values, M_values)

direct = N * M
onnx_hub = N + M
savings_ratio = direct / onnx_hub

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Direct converters needed
im1 = axes[0].imshow(direct, origin='lower', cmap='Reds', aspect='auto',
                      extent=[2, 14, 2, 14])
axes[0].set_xlabel('Number of Frameworks (N)')
axes[0].set_ylabel('Number of Runtimes (M)')
axes[0].set_title('Direct: N×M Converters')
plt.colorbar(im1, ax=axes[0])

# Plot 2: ONNX hub converters
im2 = axes[1].imshow(onnx_hub, origin='lower', cmap='Greens', aspect='auto',
                      extent=[2, 14, 2, 14])
axes[1].set_xlabel('Number of Frameworks (N)')
axes[1].set_ylabel('Number of Runtimes (M)')
axes[1].set_title('ONNX Hub: N+M Converters')
plt.colorbar(im2, ax=axes[1])

# Plot 3: Savings ratio
im3 = axes[2].imshow(savings_ratio, origin='lower', cmap='Blues', aspect='auto',
                      extent=[2, 14, 2, 14])
axes[2].set_xlabel('Number of Frameworks (N)')
axes[2].set_ylabel('Number of Runtimes (M)')
axes[2].set_title('Savings Ratio: NM/(N+M)')
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.suptitle('The N×M Interoperability Problem', y=1.02, fontsize=14, fontweight='bold')
plt.show()

print(f"Example: N=5 frameworks, M=8 runtimes")
print(f"  Direct converters needed: {5*8} = 40")
print(f"  ONNX hub converters:      {5+8} = 13")
print(f"  Savings ratio:            {5*8/(5+8):.1f}×")

<a id='section-2'></a>
## Section 2: Historical Context and Motivation

### Timeline of ML Framework Fragmentation

The deep learning revolution (2012–2017) produced a proliferation of frameworks, each with its own internal representation of neural networks. This fragmentation created enormous friction in the ML pipeline:

```
Timeline of Major ML Frameworks and ONNX:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2012 ─── AlexNet / Caffe emerges
2015 ─── TensorFlow 1.0 released (Google)
2016 ─── PyTorch 0.1 (Facebook/Meta)
2017 ─── ONNX announced (Facebook + Microsoft)   ◀━━━ KEY EVENT
         ├── Initial spec: ~80 operators
         └── Goal: "train once, deploy anywhere"
2018 ─── ONNX Runtime (ORT) released by Microsoft
2019 ─── ONNX joins Linux Foundation (LF AI)
2020 ─── OpSet 13, dynamic shapes mature
2021 ─── ONNX graduates to LF AI & Data
2022 ─── PyTorch 2.0 with torch.onnx improvements
2023 ─── OpSet 20+, transformer op patterns
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
```

### The Core Motivation

Before ONNX, deploying a model trained in PyTorch to a mobile device running CoreML required either (a) re-implementing the model in Swift/Objective-C, (b) maintaining a custom conversion script, or (c) running a Python server. None of these scaled. The fundamental insight behind ONNX was that **all neural networks, regardless of the framework used to define them, share the same mathematical semantics**: they are compositions of differentiable operators applied to tensors.

If we could define a **canonical, framework-agnostic representation** of these operations, then any framework could export to this format, and any runtime could import from it. This is precisely the role ONNX fills: a **lingua franca** for neural network computation graphs.

### Design Principles

The ONNX specification was designed around several key principles:

1. **Completeness**: The operator set must cover the vast majority of operations used in practice
2. **Extensibility**: Custom operators can be defined without modifying the core spec
3. **Versioning**: Both the IR and individual operators are versioned for backward compatibility
4. **Efficiency**: The serialization format (Protocol Buffers) supports efficient parsing and memory mapping
5. **Determinism**: Given the same inputs, an ONNX model must produce the same outputs (modulo floating-point non-determinism)

<a id='section-3'></a>
## Section 3: Formal Definition of ONNX

### Definition 3.1 (ONNX Model)

An ONNX model $\mathcal{M}$ is a tuple:

$$\mathcal{M} = (\mathcal{G}, \mathcal{O}, \mathcal{V}, \mathcal{P})$$

where:
- $\mathcal{G}$ is the **computation graph** (a directed acyclic graph)
- $\mathcal{O}$ is the **operator set** (opset) — a versioned collection of operator definitions
- $\mathcal{V}$ is the **IR version** — the version of the intermediate representation specification
- $\mathcal{P}$ is the **metadata** — producer name, domain, doc strings, etc.

### Definition 3.2 (Computation Graph)

The computation graph $\mathcal{G} = (V, E, W)$ consists of:
- $V = \{v_1, v_2, \ldots, v_n\}$: a set of **nodes** (operator invocations)
- $E \subseteq V \times V$: a set of directed **edges** (data flow connections)
- $W = \{w_1, w_2, \ldots, w_k\}$: a set of **initializers** (constant tensors / learned parameters)

Each node $v_i$ is characterized by:
$$v_i = (\text{op\_type}_i, \text{inputs}_i, \text{outputs}_i, \text{attributes}_i)$$

### Definition 3.3 (Tensor)

A tensor $T$ in ONNX is a multi-dimensional array with:
- **Name**: A unique string identifier within the graph
- **Element type**: $\tau \in \{\text{float16}, \text{float32}, \text{float64}, \text{int8}, \ldots, \text{string}\}$
- **Shape**: $S = (d_1, d_2, \ldots, d_r)$ where $r$ is the rank and each $d_i \in \mathbb{N} \cup \{\text{symbolic}\}$

Formally: $T \in \mathbb{T}^{d_1 \times d_2 \times \cdots \times d_r}$ where $\mathbb{T}$ is the element type's domain.

### The Protobuf Hierarchy

```
┌─────────────────────────────────────────────────────────────────┐
│                        ModelProto                                │
├─────────────────────────────────────────────────────────────────┤
│  ir_version: int64                                              │
│  opset_import: [OperatorSetIdProto]                             │
│  producer_name: string                                          │
│  ┌─────────────────────────────────────────────────────────┐   │
│  │                     GraphProto                           │   │
│  ├─────────────────────────────────────────────────────────┤   │
│  │  name: string                                           │   │
│  │  input: [ValueInfoProto]                                │   │
│  │  output: [ValueInfoProto]                               │   │
│  │  initializer: [TensorProto]                             │   │
│  │  ┌───────────────────────────────────────────────────┐  │   │
│  │  │              NodeProto (repeated)                  │  │   │
│  │  ├───────────────────────────────────────────────────┤  │   │
│  │  │  op_type: string (e.g., "MatMul", "Conv")        │  │   │
│  │  │  input: [string]                                  │  │   │
│  │  │  output: [string]                                 │  │   │
│  │  │  attribute: [AttributeProto]                      │  │   │
│  │  └───────────────────────────────────────────────────┘  │   │
│  └─────────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Install dependencies
!pip install onnx numpy matplotlib networkx -q

In [ ]:
import onnx
from onnx import helper, TensorProto
from onnx.checker import check_model
import numpy as np

print(f"ONNX version: {onnx.__version__}")
print(f"IR version supported: {onnx.IR_VERSION}")
print(f"\nONNX Tensor Data Types:")
print(f"{'Type':<15} {'Code':<6} {'NumPy Equivalent'}")
print("─" * 45)
type_map = {
    'FLOAT': TensorProto.FLOAT,
    'DOUBLE': TensorProto.DOUBLE,
    'FLOAT16': TensorProto.FLOAT16,
    'INT8': TensorProto.INT8,
    'INT16': TensorProto.INT16,
    'INT32': TensorProto.INT32,
    'INT64': TensorProto.INT64,
    'UINT8': TensorProto.UINT8,
    'BOOL': TensorProto.BOOL,
    'STRING': TensorProto.STRING,
}
np_equiv = {
    'FLOAT': 'float32', 'DOUBLE': 'float64', 'FLOAT16': 'float16',
    'INT8': 'int8', 'INT16': 'int16', 'INT32': 'int32',
    'INT64': 'int64', 'UINT8': 'uint8', 'BOOL': 'bool', 'STRING': 'object'
}
for name, code in type_map.items():
    print(f"{name:<15} {code:<6} {np_equiv[name]}")

<a id='section-4'></a>
## Section 4: The Computation Graph — Mathematical Foundation

### Definition 4.1 (Directed Acyclic Graph)

An ONNX computation graph is formally a **DAG** $G = (V, E)$ where:

$$V = \{v_1, v_2, \ldots, v_n\} \quad \text{(operator nodes)}$$
$$E = \{(v_i, v_j) \mid v_i \text{ produces a tensor consumed by } v_j\}$$

The **acyclicity constraint** ensures that there exists a topological ordering $\sigma: V \to \{1, 2, \ldots, n\}$ such that:

$$\forall (v_i, v_j) \in E: \sigma(v_i) < \sigma(v_j)$$

This ordering guarantees that every node's inputs are available before it executes.

### Definition 4.2 (Operator Semantics)

Each node $v_i$ with operator type $\text{op}$ computes a function:

$$f_{\text{op}}: \underbrace{\mathbb{T}_1 \times \mathbb{T}_2 \times \cdots \times \mathbb{T}_k}_{\text{inputs}} \to \underbrace{\mathbb{T}'_1 \times \mathbb{T}'_2 \times \cdots \times \mathbb{T}'_l}_{\text{outputs}}$$

where each $\mathbb{T}_i$ is a tensor space of specific type and shape.

### Example: Linear Layer Computation

A linear layer $y = Wx + b$ where $W \in \mathbb{R}^{m \times n}$, $x \in \mathbb{R}^{n}$, $b \in \mathbb{R}^{m}$ decomposes into:

$$v_1: \text{MatMul}(W, x) \to h \in \mathbb{R}^{m}$$
$$v_2: \text{Add}(h, b) \to y \in \mathbb{R}^{m}$$

With the edge $(v_1, v_2)$ representing the flow of tensor $h$.

### Topological Execution

The execution of an ONNX graph follows the topological sort. For a graph with $n$ nodes, the execution trace is:

$$\text{Execute}(G, \text{inputs}) = f_{v_{\sigma^{-1}(n)}} \circ \cdots \circ f_{v_{\sigma^{-1}(2)}} \circ f_{v_{\sigma^{-1}(1)}}(\text{inputs})$$

where $\sigma^{-1}(k)$ gives the node at position $k$ in the topological order. Note that the composition is not strictly sequential — nodes without data dependencies may execute in parallel.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

# Visualize a computation graph for y = relu(Wx + b)
G = nx.DiGraph()

# Add nodes with types
nodes = {
    'W': {'type': 'initializer', 'shape': '(m,n)'},
    'x': {'type': 'input', 'shape': '(n,)'},
    'b': {'type': 'initializer', 'shape': '(m,)'},
    'MatMul': {'type': 'operator', 'shape': ''},
    'Add': {'type': 'operator', 'shape': ''},
    'Relu': {'type': 'operator', 'shape': ''},
    'y': {'type': 'output', 'shape': '(m,)'},
}

for name, attrs in nodes.items():
    G.add_node(name, **attrs)

edges = [
    ('W', 'MatMul'), ('x', 'MatMul'),
    ('MatMul', 'Add'), ('b', 'Add'),
    ('Add', 'Relu'), ('Relu', 'y')
]
G.add_edges_from(edges)

# Layout
pos = {
    'W': (0, 2), 'x': (2, 2), 'b': (4, 2),
    'MatMul': (1, 1), 'Add': (3, 1),
    'Relu': (2, 0), 'y': (2, -1)
}

fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# Color by type
color_map = {'initializer': '#FF9999', 'input': '#99FF99', 'operator': '#9999FF', 'output': '#FFFF99'}
node_colors = [color_map[nodes[n]['type']] for n in G.nodes()]

nx.draw(G, pos, ax=ax, with_labels=True, node_color=node_colors,
        node_size=2500, font_size=11, font_weight='bold',
        arrows=True, arrowsize=20, edge_color='gray',
        connectionstyle='arc3,rad=0.1')

# Add edge labels (tensor names)
edge_labels = {
    ('W', 'MatMul'): 'W∈ℝ^(m×n)',
    ('x', 'MatMul'): 'x∈ℝⁿ',
    ('MatMul', 'Add'): 'h∈ℝᵐ',
    ('b', 'Add'): 'b∈ℝᵐ',
    ('Add', 'Relu'): 'z∈ℝᵐ',
    ('Relu', 'y'): 'y∈ℝᵐ',
}
nx.draw_networkx_edge_labels(G, pos, edge_labels, ax=ax, font_size=8)

# Legend
legend_elements = [
    mpatches.Patch(color='#FF9999', label='Initializer (weights)'),
    mpatches.Patch(color='#99FF99', label='Input tensor'),
    mpatches.Patch(color='#9999FF', label='Operator node'),
    mpatches.Patch(color='#FFFF99', label='Output tensor'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=10)
ax.set_title('ONNX Computation Graph: $y = \\text{ReLU}(Wx + b)$', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

<a id='section-5'></a>
## Section 5: ONNX as a Universal Intermediate Representation

### The IR Analogy: Compilers and ONNX

ONNX plays the same architectural role in ML that **LLVM IR** plays in compiler design. Just as LLVM provides a common intermediate representation between source languages (C, C++, Rust) and target architectures (x86, ARM, RISC-V), ONNX provides a common representation between training frameworks and inference engines:

```
COMPILER ANALOGY:                         ML ANALOGY:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Source Languages    ─── Frontend ──▶      Training Frameworks  ─── Exporter ──▶
(C, C++, Rust)                            (PyTorch, TF, JAX)
         │                                         │
         ▼                                         ▼
    LLVM IR                                    ONNX Format
  (universal)                                 (universal)
         │                                         │
         ▼                                         ▼
Target Backends    ◀── Backend ───          Inference Runtimes  ◀── Importer ───
(x86, ARM, GPU)                             (ORT, TensorRT, OpenVINO)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
```

### Properties of a Good IR

A well-designed intermediate representation must satisfy:

1. **Semantic Preservation**: $\forall x: f_{\text{original}}(x) \approx f_{\text{ONNX}}(x)$ (up to floating-point tolerance)
2. **Optimization Opportunity**: The IR should expose patterns that enable graph-level optimizations (fusion, constant folding, dead code elimination)
3. **Target Independence**: The IR should not embed assumptions about specific hardware
4. **Roundtrip Fidelity**: Export → Import should preserve model semantics

### The Semantic Gap

Different frameworks represent the same mathematical operation differently. For example, a PyTorch `nn.Linear(in_features=768, out_features=3072)` might be represented as:

- **PyTorch**: A module with `.weight` (transposed) and `.bias` attributes
- **TensorFlow**: A `Dense` layer with kernel and bias variables in a scope
- **ONNX**: A `Gemm` node with attributes `alpha=1.0`, `beta=1.0`, `transB=1`

The ONNX exporter's job is to bridge this semantic gap — translating framework-specific idioms into the canonical ONNX operator vocabulary while preserving numerical equivalence.

In [ ]:
# Demonstrate building an ONNX model from scratch — a 2-layer MLP
# y = ReLU(W1 @ x + b1) @ W2 + b2

import numpy as np
from onnx import helper, TensorProto, numpy_helper
from onnx.checker import check_model

# Define dimensions
input_dim = 4
hidden_dim = 8
output_dim = 3
batch_size = 1

# Create random weights (initializers)
np.random.seed(42)
W1_data = np.random.randn(input_dim, hidden_dim).astype(np.float32)
b1_data = np.random.randn(hidden_dim).astype(np.float32)
W2_data = np.random.randn(hidden_dim, output_dim).astype(np.float32)
b2_data = np.random.randn(output_dim).astype(np.float32)

# Convert to ONNX tensors
W1 = numpy_helper.from_array(W1_data, name='W1')
b1 = numpy_helper.from_array(b1_data, name='b1')
W2 = numpy_helper.from_array(W2_data, name='W2')
b2 = numpy_helper.from_array(b2_data, name='b2')

# Define graph nodes
matmul1 = helper.make_node('MatMul', ['X', 'W1'], ['h1_pre'])
add1 = helper.make_node('Add', ['h1_pre', 'b1'], ['h1_bias'])
relu = helper.make_node('Relu', ['h1_bias'], ['h1'])
matmul2 = helper.make_node('MatMul', ['h1', 'W2'], ['h2_pre'])
add2 = helper.make_node('Add', ['h2_pre', 'b2'], ['Y'])

# Define input/output
X_info = helper.make_tensor_value_info('X', TensorProto.FLOAT, [batch_size, input_dim])
Y_info = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [batch_size, output_dim])

# Build graph
graph = helper.make_graph(
    nodes=[matmul1, add1, relu, matmul2, add2],
    name='two_layer_mlp',
    inputs=[X_info],
    outputs=[Y_info],
    initializer=[W1, b1, W2, b2]
)

# Build model
model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])
model.ir_version = 8
check_model(model)

print("✓ Model validated successfully!")
print(f"\nModel Structure:")
print(f"  IR Version: {model.ir_version}")
print(f"  OpSet: {model.opset_import[0].version}")
print(f"  Graph: '{model.graph.name}'")
print(f"  Nodes: {len(model.graph.node)}")
print(f"  Initializers: {len(model.graph.initializer)}")
print(f"\nComputation Flow:")
for i, node in enumerate(model.graph.node):
    print(f"  [{i}] {node.op_type}: {list(node.input)} → {list(node.output)}")

<a id='section-6'></a>
## Section 6: Visualization — The Framework-Runtime Connectivity Matrix

### Quantifying Ecosystem Coverage

Let us define the **coverage matrix** $C \in \{0, 1\}^{N \times M}$ where:

$$C_{ij} = \begin{cases} 1 & \text{if framework } f_i \text{ can export to runtime } d_j \text{ via ONNX} \\ 0 & \text{otherwise} \end{cases}$$

The **interoperability score** of the ecosystem is:

$$\text{Score} = \frac{\sum_{i,j} C_{ij}}{N \cdot M} \in [0, 1]$$

A score of 1.0 means every framework can deploy to every runtime — the ideal state that ONNX strives toward.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Framework-Runtime compatibility matrix via ONNX
frameworks = ['PyTorch', 'TensorFlow', 'JAX', 'scikit-learn', 'XGBoost', 'LightGBM']
runtimes = ['ONNX Runtime', 'TensorRT', 'OpenVINO', 'CoreML', 'NNAPI', 'DirectML', 'WebNN']

# Coverage matrix (1 = supported via ONNX path)
coverage = np.array([
    [1, 1, 1, 1, 1, 1, 1],  # PyTorch
    [1, 1, 1, 1, 1, 1, 1],  # TensorFlow
    [1, 1, 1, 0, 0, 1, 1],  # JAX
    [1, 0, 1, 0, 0, 1, 0],  # scikit-learn
    [1, 0, 1, 0, 0, 1, 0],  # XGBoost
    [1, 0, 1, 0, 0, 1, 0],  # LightGBM
])

fig, ax = plt.subplots(figsize=(12, 7))
im = ax.imshow(coverage, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)

ax.set_xticks(range(len(runtimes)))
ax.set_yticks(range(len(frameworks)))
ax.set_xticklabels(runtimes, rotation=45, ha='right', fontsize=11)
ax.set_yticklabels(frameworks, fontsize=11)

# Annotate cells
for i in range(len(frameworks)):
    for j in range(len(runtimes)):
        text = '✓' if coverage[i, j] else '✗'
        color = 'darkgreen' if coverage[i, j] else 'darkred'
        ax.text(j, i, text, ha='center', va='center', fontsize=16, color=color, fontweight='bold')

ax.set_title('ONNX Interoperability Matrix: Framework → Runtime Coverage', fontsize=13, fontweight='bold')
ax.set_xlabel('Inference Runtimes (Deployment Targets)', fontsize=11)
ax.set_ylabel('Training Frameworks (Sources)', fontsize=11)

N, M = coverage.shape
score = coverage.sum() / (N * M)
ax.text(0.5, -0.18, f'Interoperability Score = {coverage.sum()}/{N*M} = {score:.2f}',
        transform=ax.transAxes, ha='center', fontsize=12,
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

plt.tight_layout()
plt.show()

<a id='section-7'></a>
## Section 7: Worked Example — Step-by-Step Graph Construction

### Problem Statement

Construct the ONNX graph for a **softmax classifier**:

$$\hat{y} = \text{Softmax}(Wx + b)$$

where:
- $x \in \mathbb{R}^{d}$ is the input feature vector
- $W \in \mathbb{R}^{c \times d}$ is the weight matrix ($c$ classes)
- $b \in \mathbb{R}^{c}$ is the bias vector
- $\hat{y} \in \Delta^{c-1}$ is the output probability simplex

### Step 1: Identify Operators

The computation decomposes as:
1. $h = W \cdot x$ → `MatMul` operator
2. $z = h + b$ → `Add` operator  
3. $\hat{y} = \text{softmax}(z)$ → `Softmax` operator

### Step 2: Define the Softmax

The Softmax function maps $\mathbb{R}^c \to \Delta^{c-1}$ (the probability simplex):

$$\text{Softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{c} e^{z_j}} \quad \forall i \in \{1, \ldots, c\}$$

Properties:
- $\sum_i \hat{y}_i = 1$ (normalization)
- $\hat{y}_i > 0 \; \forall i$ (positivity)
- $\frac{\partial \hat{y}_i}{\partial z_j} = \hat{y}_i(\delta_{ij} - \hat{y}_j)$ (Jacobian)

### Step 3: Shape Inference Chain

$$x \in \mathbb{R}^{1 \times d} \xrightarrow{\text{MatMul}(W^T)} h \in \mathbb{R}^{1 \times c} \xrightarrow{\text{Add}(b)} z \in \mathbb{R}^{1 \times c} \xrightarrow{\text{Softmax}} \hat{y} \in \mathbb{R}^{1 \times c}$$

In [ ]:
import numpy as np
from onnx import helper, TensorProto, numpy_helper
from onnx.checker import check_model

# Softmax classifier: y_hat = Softmax(x @ W + b)
d = 4   # input features
c = 3   # number of classes

# Step 1: Create weight initializers
np.random.seed(123)
W_data = np.random.randn(d, c).astype(np.float32) * 0.1
b_data = np.zeros(c, dtype=np.float32)

W_init = numpy_helper.from_array(W_data, name='W')
b_init = numpy_helper.from_array(b_data, name='b')

# Step 2: Define nodes (operators)
matmul_node = helper.make_node('MatMul', inputs=['X', 'W'], outputs=['H'])
add_node = helper.make_node('Add', inputs=['H', 'b'], outputs=['Z'])
softmax_node = helper.make_node('Softmax', inputs=['Z'], outputs=['Y_hat'], axis=1)

# Step 3: Define input/output value info
X_info = helper.make_tensor_value_info('X', TensorProto.FLOAT, [1, d])
Y_info = helper.make_tensor_value_info('Y_hat', TensorProto.FLOAT, [1, c])

# Step 4: Assemble graph
graph = helper.make_graph(
    nodes=[matmul_node, add_node, softmax_node],
    name='softmax_classifier',
    inputs=[X_info],
    outputs=[Y_info],
    initializer=[W_init, b_init]
)

# Step 5: Create model
model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])
check_model(model)

# Verify with manual computation
x_test = np.array([[1.0, 2.0, 3.0, 4.0]], dtype=np.float32)
h_manual = x_test @ W_data
z_manual = h_manual + b_data
exp_z = np.exp(z_manual - z_manual.max(axis=1, keepdims=True))  # numerical stability
y_hat_manual = exp_z / exp_z.sum(axis=1, keepdims=True)

print("Worked Example: Softmax Classifier")
print("=" * 50)
print(f"\nInput x shape: {x_test.shape} → ℝ^(1×{d})")
print(f"Weight W shape: {W_data.shape} → ℝ^({d}×{c})")
print(f"Bias b shape: {b_data.shape} → ℝ^{c}")
print(f"\nStep 1 — MatMul: x @ W = h")
print(f"  h = {h_manual.round(4)}")
print(f"  shape: (1,{d}) × ({d},{c}) → (1,{c})")
print(f"\nStep 2 — Add: h + b = z")
print(f"  z = {z_manual.round(4)}")
print(f"\nStep 3 — Softmax: softmax(z) = ŷ")
print(f"  ŷ = {y_hat_manual.round(4)}")
print(f"  sum(ŷ) = {y_hat_manual.sum():.6f} (should be 1.0)")
print(f"  Predicted class: {np.argmax(y_hat_manual)}")

<a id='section-8'></a>
## Section 8: The ONNX Operator Algebra

### Definition 8.1 (Operator Set)

An ONNX **Operator Set** (OpSet) is a versioned collection of operator definitions:

$$\mathcal{O}_v = \{(\text{name}_i, \text{spec}_i, \text{version}_i) \mid i = 1, \ldots, |\mathcal{O}_v|\}$$

where $v$ is the opset version number. Each operator specification defines:
- **Input constraints**: number, types, and shapes of inputs
- **Output specification**: number, types, and shapes of outputs
- **Attributes**: compile-time constants (e.g., kernel_shape for Conv)
- **Type constraints**: polymorphic type variables (e.g., `T` = float | double)

### Operator Categories

The ONNX operator set is organized into functional categories:

| Category | Examples | Count |
|----------|----------|-------|
| **Arithmetic** | Add, Sub, Mul, Div, Pow | ~10 |
| **Matrix** | MatMul, Gemm, Einsum | ~5 |
| **Activation** | Relu, Sigmoid, Tanh, Softmax | ~15 |
| **Convolution** | Conv, ConvTranspose | ~3 |
| **Pooling** | MaxPool, AveragePool, GlobalAveragePool | ~5 |
| **Normalization** | BatchNormalization, LayerNormalization | ~5 |
| **Reduction** | ReduceSum, ReduceMean, ReduceMax | ~10 |
| **Shape** | Reshape, Transpose, Squeeze, Unsqueeze | ~15 |
| **Logic** | Equal, Greater, Less, Where | ~10 |
| **Tensor** | Concat, Split, Gather, Scatter | ~15 |

### Versioning Semantics

ONNX uses a **monotonic versioning** scheme:

$$\text{opset\_version}(\text{op}) = \max\{v \mid \text{op is defined in } \mathcal{O}_v\}$$

Each new opset version may:
1. **Add** new operators
2. **Update** existing operators (new attributes, extended type support)
3. **Deprecate** operators (but never remove for backward compatibility)

The compatibility guarantee:
$$\text{opset } v_1 \leq v_2 \implies \text{models using } \mathcal{O}_{v_1} \text{ are valid under } \mathcal{O}_{v_2}$$

In [ ]:
import onnx
from onnx import defs
import matplotlib.pyplot as plt
from collections import Counter

# Explore the ONNX operator registry
all_schemas = defs.get_all_schemas_with_history()

# Get latest version of each operator
latest_ops = {}
for schema in all_schemas:
    name = schema.name
    domain = schema.domain
    if domain == '':
        if name not in latest_ops or schema.since_version > latest_ops[name].since_version:
            latest_ops[name] = schema

print(f"Total unique operators in default domain: {len(latest_ops)}")
print(f"Total schemas (including versions): {len(all_schemas)}")

# Categorize operators by their characteristics
categories = {
    'Arithmetic': ['Add', 'Sub', 'Mul', 'Div', 'Pow', 'Mod', 'Neg', 'Abs', 'Sqrt', 'Exp', 'Log'],
    'Matrix': ['MatMul', 'Gemm', 'Einsum', 'MatMulInteger'],
    'Activation': ['Relu', 'Sigmoid', 'Tanh', 'Softmax', 'LeakyRelu', 'Elu', 'Selu', 'Gelu'],
    'Convolution': ['Conv', 'ConvTranspose', 'ConvInteger'],
    'Pooling': ['MaxPool', 'AveragePool', 'GlobalAveragePool', 'GlobalMaxPool'],
    'Normalization': ['BatchNormalization', 'LayerNormalization', 'InstanceNormalization'],
    'Reduction': ['ReduceSum', 'ReduceMean', 'ReduceMax', 'ReduceMin', 'ReduceProd'],
    'Shape/Tensor': ['Reshape', 'Transpose', 'Squeeze', 'Unsqueeze', 'Concat', 'Split',
                     'Gather', 'Scatter', 'Flatten', 'Expand', 'Tile'],
    'Comparison': ['Equal', 'Greater', 'Less', 'Where', 'And', 'Or', 'Not'],
}

cat_counts = {k: len([op for op in v if op in latest_ops]) for k, v in categories.items()}
cat_counts['Other'] = len(latest_ops) - sum(cat_counts.values())

# Plot operator distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart of categories
colors = plt.cm.Set3(np.linspace(0, 1, len(cat_counts)))
wedges, texts, autotexts = ax1.pie(
    cat_counts.values(), labels=cat_counts.keys(), autopct='%1.0f%%',
    colors=colors, textprops={'fontsize': 9}
)
ax1.set_title('ONNX Operator Distribution by Category', fontsize=12, fontweight='bold')

# Version introduction histogram
since_versions = [schema.since_version for schema in latest_ops.values()]
version_counts = Counter(since_versions)
versions = sorted(version_counts.keys())
counts = [version_counts[v] for v in versions]

ax2.bar(versions, counts, color='steelblue', edgecolor='navy', alpha=0.8)
ax2.set_xlabel('OpSet Version (since_version)', fontsize=11)
ax2.set_ylabel('Number of Operators Introduced', fontsize=11)
ax2.set_title('Operators Introduced per OpSet Version', fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

<a id='section-9'></a>
## Section 9: Key Properties and Guarantees

### Property 9.1: Deterministic Evaluation

For a well-formed ONNX model $\mathcal{M}$ with graph $G$ and inputs $\mathbf{x}$:

$$\text{eval}(G, \mathbf{x}) = \text{eval}(G, \mathbf{x}) \quad \text{(deterministic, same hardware)}$$

However, across hardware:
$$\|\text{eval}_{\text{CPU}}(G, \mathbf{x}) - \text{eval}_{\text{GPU}}(G, \mathbf{x})\| \leq \epsilon_{\text{fp}}$$

where $\epsilon_{\text{fp}}$ accounts for floating-point non-associativity.

### Property 9.2: Graph Equivalence Under Optimization

A graph optimization $\phi: G \to G'$ is **semantically valid** if:

$$\forall \mathbf{x} \in \text{dom}(G): \|\text{eval}(G, \mathbf{x}) - \text{eval}(G', \mathbf{x})\|_\infty \leq \epsilon$$

Common valid optimizations:
- **Constant folding**: Pre-computing static subgraphs
- **Operator fusion**: Combining adjacent operators (e.g., Conv + BN)
- **Dead code elimination**: Removing unreachable nodes

### Property 9.3: Backward Compatibility

ONNX maintains **strong backward compatibility**:

$$\text{valid}(\mathcal{M}, \text{opset}_v) \implies \text{valid}(\mathcal{M}, \text{opset}_{v'}) \quad \forall v' \geq v$$

This means a model exported with opset 13 will remain valid and produce correct results with any runtime supporting opset 13 or higher.

### Property 9.4: Type Safety

ONNX enforces **static type safety** through type and shape inference. For every edge in the graph carrying tensor $t$:

$$\text{type}(t) \in \text{allowed\_types}(\text{consumer\_input})$$

This is verified at model validation time by the `onnx.checker` module.

In [ ]:
# Demonstrate type checking and validation
from onnx import helper, TensorProto, checker, shape_inference
import traceback

# Example 1: Valid model
print("Example 1: Valid model construction")
print("=" * 50)

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [2, 3])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [2, 3])
node = helper.make_node('Relu', ['X'], ['Y'])
graph = helper.make_graph([node], 'valid_graph', [X], [Y])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])

try:
    checker.check_model(model)
    print("✓ Model is valid!")
except Exception as e:
    print(f"✗ Validation error: {e}")

# Example 2: Shape inference
print("\nExample 2: Automatic shape inference")
print("=" * 50)

# Build a model without specifying intermediate shapes
A_info = helper.make_tensor_value_info('A', TensorProto.FLOAT, [2, 4])
B_info = helper.make_tensor_value_info('B', TensorProto.FLOAT, [4, 3])
C_info = helper.make_tensor_value_info('C', TensorProto.FLOAT, None)  # shape unknown

matmul = helper.make_node('MatMul', ['A', 'B'], ['C'])
graph2 = helper.make_graph([matmul], 'shape_test', [A_info, B_info], [C_info])
model2 = helper.make_model(graph2, opset_imports=[helper.make_opsetid('', 17)])

# Run shape inference
inferred = shape_inference.infer_shapes(model2)
output_shape = inferred.graph.output[0].type.tensor_type.shape
dims = [d.dim_value for d in output_shape.dim]
print(f"Input A: [2, 4]  (ℝ^(2×4))")
print(f"Input B: [4, 3]  (ℝ^(4×3))")
print(f"Output C (inferred): {dims}  (ℝ^(2×3))")
print(f"\nShape inference rule: MatMul(ℝ^(m×k), ℝ^(k×n)) → ℝ^(m×n)")
print(f"Applied: MatMul(ℝ^(2×4), ℝ^(4×3)) → ℝ^(2×3) ✓")

In [ ]:
# Visualize the protobuf structure of our model as a tree
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')

# Draw the protobuf hierarchy as a tree diagram
def draw_box(ax, x, y, w, h, text, color, fontsize=9):
    rect = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                          facecolor=color, edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold')

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

# Level 0: ModelProto
draw_box(ax, 5, 8.5, 4, 0.8, 'ModelProto', '#FFD700', 11)

# Level 1: Model attributes
draw_box(ax, 0.5, 6.8, 2.5, 0.6, 'ir_version=8', '#FFE4B5', 8)
draw_box(ax, 3.5, 6.8, 3, 0.6, 'opset_import=[v17]', '#FFE4B5', 8)
draw_box(ax, 7, 6.8, 2.5, 0.6, 'GraphProto', '#87CEEB', 10)
draw_box(ax, 10, 6.8, 3, 0.6, 'producer="onnx"', '#FFE4B5', 8)

draw_arrow(ax, 7, 8.5, 1.75, 7.4)
draw_arrow(ax, 7, 8.5, 5, 7.4)
draw_arrow(ax, 7, 8.5, 8.25, 7.4)
draw_arrow(ax, 7, 8.5, 11.5, 7.4)

# Level 2: Graph components
draw_box(ax, 1, 5, 2, 0.6, 'inputs: [X]', '#98FB98', 8)
draw_box(ax, 3.5, 5, 2.2, 0.6, 'outputs: [Y]', '#98FB98', 8)
draw_box(ax, 6.2, 5, 2.8, 0.6, 'initializers: [W,b]', '#DDA0DD', 8)
draw_box(ax, 9.5, 5, 2.5, 0.6, 'nodes: [...]', '#FFA07A', 9)

draw_arrow(ax, 8.25, 6.8, 2, 5.6)
draw_arrow(ax, 8.25, 6.8, 4.6, 5.6)
draw_arrow(ax, 8.25, 6.8, 7.6, 5.6)
draw_arrow(ax, 8.25, 6.8, 10.75, 5.6)

# Level 3: Node details
draw_box(ax, 1, 3.2, 3, 0.6, 'MatMul(X,W)→H', '#FFA07A', 8)
draw_box(ax, 4.5, 3.2, 3, 0.6, 'Add(H,b)→Z', '#FFA07A', 8)
draw_box(ax, 8.5, 3.2, 3.5, 0.6, 'Softmax(Z)→Y_hat', '#FFA07A', 8)

draw_arrow(ax, 10.75, 5, 2.5, 3.8)
draw_arrow(ax, 10.75, 5, 6, 3.8)
draw_arrow(ax, 10.75, 5, 10.25, 3.8)

# Level 4: Value info
draw_box(ax, 0.5, 1.5, 3.5, 0.6, 'ValueInfo: X [1,4] float32', '#98FB98', 7)
draw_box(ax, 4.5, 1.5, 4, 0.6, 'TensorProto: W [4,3] float32', '#DDA0DD', 7)
draw_box(ax, 9, 1.5, 4, 0.6, 'ValueInfo: Y_hat [1,3] float32', '#98FB98', 7)

# Title
ax.set_title('ONNX Model Protobuf Structure (Softmax Classifier)', fontsize=14, fontweight='bold', pad=20)

# Legend
legend_elements = [
    mpatches.Patch(color='#FFD700', label='ModelProto (root)'),
    mpatches.Patch(color='#87CEEB', label='GraphProto'),
    mpatches.Patch(color='#FFA07A', label='NodeProto'),
    mpatches.Patch(color='#98FB98', label='ValueInfoProto'),
    mpatches.Patch(color='#DDA0DD', label='TensorProto (initializer)'),
    mpatches.Patch(color='#FFE4B5', label='Scalar attributes'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.show()

<a id='section-10'></a>
## Section 10: Summary and Connections

### Core Takeaways

1. **ONNX solves the $N \times M$ problem** by serving as a hub format, reducing converter complexity from $O(NM)$ to $O(N + M)$

2. **Formally, an ONNX model is a tuple** $\mathcal{M} = (\mathcal{G}, \mathcal{O}, \mathcal{V}, \mathcal{P})$ containing a computation graph, operator set, IR version, and metadata

3. **The computation graph is a DAG** $G = (V, E)$ with typed tensor edges and operator nodes, admitting topological execution

4. **Type safety and shape inference** are enforced statically, enabling compile-time validation before deployment

5. **Versioned operator sets** ensure backward compatibility while allowing the spec to evolve

### Mathematical Framework Summary

$$\boxed{\text{ONNX Model} = \underbrace{\text{DAG}(V, E)}_{\text{structure}} + \underbrace{\text{OpSet}_v}_{\text{semantics}} + \underbrace{\text{TypeSystem}}_{\text{safety}} + \underbrace{\text{Initializers}}_{\text{parameters}}}$$

### Connections to Subsequent Topics

| This Section | Next Sections |
|:---|:---|
| N×M problem (why ONNX exists) | → 1.2 Why ONNX Matters (deeper cost analysis) |
| Computation graph basics | → 2.1 Computation Graph Basics (formal DAG theory) |
| Operator set overview | → 2.2 Nodes, Edges, and Tensors (operator anatomy) |
| Protobuf structure intro | → 2.3 ONNX IR Specification (full proto hierarchy) |
| Type system sketch | → 2.4 Type System and Shapes (formal type inference) |

In [ ]:
# Final summary visualization: ONNX's position in the ML ecosystem
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis('off')

# Training frameworks (left)
frameworks = ['PyTorch', 'TensorFlow', 'JAX', 'scikit-learn']
for i, fw in enumerate(frameworks):
    y = 6.5 - i * 1.5
    rect = FancyBboxPatch((0.5, y), 2.5, 0.8, boxstyle='round,pad=0.1',
                          facecolor='#FF6B6B', edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(1.75, y + 0.4, fw, ha='center', va='center', fontsize=10, fontweight='bold')
    # Arrow to ONNX
    ax.annotate('', xy=(5.3, 4.4), xytext=(3.0, y + 0.4),
                arrowprops=dict(arrowstyle='->', color='#666', lw=1.2))

# ONNX hub (center)
onnx_rect = FancyBboxPatch((5, 3.5), 3.5, 1.8, boxstyle='round,pad=0.2',
                            facecolor='#4ECDC4', edgecolor='black', linewidth=2.5)
ax.add_patch(onnx_rect)
ax.text(6.75, 4.7, 'ONNX', ha='center', va='center', fontsize=16, fontweight='bold')
ax.text(6.75, 4.1, 'Universal IR', ha='center', va='center', fontsize=10, style='italic')
ax.text(6.75, 3.7, '.onnx protobuf', ha='center', va='center', fontsize=8, color='#333')

# Runtimes (right)
runtimes = ['ONNX Runtime', 'TensorRT', 'OpenVINO', 'CoreML']
for i, rt in enumerate(runtimes):
    y = 6.5 - i * 1.5
    rect = FancyBboxPatch((10.5, y), 2.8, 0.8, boxstyle='round,pad=0.1',
                          facecolor='#45B7D1', edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(11.9, y + 0.4, rt, ha='center', va='center', fontsize=10, fontweight='bold')
    # Arrow from ONNX
    ax.annotate('', xy=(10.5, y + 0.4), xytext=(8.5, 4.4),
                arrowprops=dict(arrowstyle='->', color='#666', lw=1.2))

# Labels
ax.text(1.75, 7.5, 'Training Frameworks', ha='center', fontsize=12, fontweight='bold', color='#CC0000')
ax.text(11.9, 7.5, 'Inference Runtimes', ha='center', fontsize=12, fontweight='bold', color='#0066CC')
ax.text(6.75, 2.8, 'N exporters + M importers = N+M converters', ha='center', fontsize=10,
        style='italic', color='#006644',
        bbox=dict(boxstyle='round', facecolor='#E8F8F5', alpha=0.8))

ax.set_title('ONNX: The Hub-and-Spoke Architecture for ML Interoperability',
             fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

---

**Next:** [What is ONNX — Apply Notebook](./What_is_ONNX_Apply.ipynb) | [Why ONNX Matters — Deep Dive](../02_Why_ONNX_Matters/Why_ONNX_Matters_Deep_Dive.ipynb)